In [27]:
from langchain.chat_models import  init_chat_model
import  warnings
warnings.filterwarnings('ignore')

In [53]:
model =  init_chat_model('ollama:gemini-3-flash-preview:cloud',temperature=0.2)

In [29]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings

In [30]:
video_id='9WSsLSq40Yw' 

try:
    ytt_api = YouTubeTranscriptApi()
    transcript_list = ytt_api.fetch(video_id)

    transcript = " ".join(chunk.text for chunk in transcript_list)
    # if you want see transcript
   # print(transcript)

except TranscriptsDisabled:
    print("not captions")

In [52]:
len(transcript)

86524

In [31]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
    )

chunks = splitter.create_documents([transcript])

In [32]:
len(chunks)

108

In [33]:
chunks[0].page_content

'- I want to jump right into\nwhat we were talking about before we started recording.\nYou said you’ve been obsessed, you’ve been running Dell for 41 years.\nYou’ve been obsessed really since you were 11 or 12,\nso you’ve been running it for 50 years, because there was\na Dell before a Dell, right? But you were saying\nsomebody was telling you this story about you in middle school. - When I was in junior high school,\nin the summers in Houston, there were classes\nat Rice University for young kids, okay? And so you could go to\nRice University to take these classes. And my mom kind of\nsigned me up, and it was great. They had these college professors,\nand they were teaching these classes. And the way I would get\nto Rice University is I would take the bus from our house, right?\nAnd it will take you downtown. - Okay.\n- And I was curious, so I was like, "Okay, if I just\nstay on the bus, takes me all the way downtown to where\nthe really tall buildings are." So, I go down there.'

In [34]:
embedding = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
)
vector_store =  FAISS.from_documents(chunks,embedding)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4321.91it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [35]:
# if you want see the list of embedding
# vector_store.index_to_docstore_id

In [38]:
retriever = vector_store.as_retriever(search_type='similarity', search_kwargs={'k':4})

In [39]:
retriever.invoke("what is the purpose of dell ?")

[Document(id='f76a181f-2d1c-4bc6-ad29-1220549e4045', metadata={}, page_content="access to incredible tools, and so we're doing this\nall over our business. And of course, our customers are\ndoing it. And what is this doing? It's unlocking the power of data, using\ncompute and using all these models, which also happens to be, you know, we're in the business of compute,\nand storage, and networking. So, it's an enormous opportunity\nfor Dell itself to be able to help all of our customers unlock\nthe power of their data. - You couldn't predict, like, the actual\ntools that were going to be invented. You just knew that\nnow we have this new... There's this new technology. That it will transform every single\nprocess inside the business. We won't be able to predict all the tools,\nbut we just know that they're coming. - If we didn't do that,\nand our competitors did, we'd be finished, all right? And so, we're not going to do that. And, you know, even if we did it"),
 Document(id='bdab19bb-8

In [41]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [56]:
question          = "how to grow and what should our goal ? if yes then what was discussed"
retrieved_docs    = retriever.invoke(question)

In [57]:
retrieved_docs

[Document(id='64290cc4-7473-464e-bbd1-aab6ba896a54', metadata={}, page_content='they\'re right, they get no work done." - Yeah, at the limit, nobody\nknows anything, right? And it\'s not a bad, philosophy,\n- Yeah. - you know, when you\'re\nthinking about the future. - It\'s good to have a\nhypothesis though, right? And ideas and experiments\nand to keep an open mind. - So, when we go back to that,\n"Hey, they\'re coming for us. They\'re going to be faster,\nmore efficient, more capable, and they\'re going to\nput us out of business. And we\'re not going to let that happen,\nbecause we\'re going to transform. We are going to be that business."\n- Right. - What was your hypothesis back then?\nDo you remember? - Oh, it was pretty simple. It\'s that it will be possible to\ndramatically improve the way we do things in all of the\ncore processes of the business, from software development,\nto support to sales, to every major function\nin our supply chain. Everything can be improved in a dra

In [58]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

'they\'re right, they get no work done." - Yeah, at the limit, nobody\nknows anything, right? And it\'s not a bad, philosophy,\n- Yeah. - you know, when you\'re\nthinking about the future. - It\'s good to have a\nhypothesis though, right? And ideas and experiments\nand to keep an open mind. - So, when we go back to that,\n"Hey, they\'re coming for us. They\'re going to be faster,\nmore efficient, more capable, and they\'re going to\nput us out of business. And we\'re not going to let that happen,\nbecause we\'re going to transform. We are going to be that business."\n- Right. - What was your hypothesis back then?\nDo you remember? - Oh, it was pretty simple. It\'s that it will be possible to\ndramatically improve the way we do things in all of the\ncore processes of the business, from software development,\nto support to sales, to every major function\nin our supply chain. Everything can be improved in a dramatic\nway, and that\'s what we\'re doing. And we\'re finding, you know,\n\nof 

In [59]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [60]:
answer = model.invoke(final_prompt)
print(answer.content)

Based on the transcript, the discussion regarding how to grow and what the goals should be includes the following:

**How to grow:**
*   **Improve core processes:** Dramatically improve every major function of the business, including software development, support, sales, and the supply chain.
*   **Negative cash conversion cycle:** Shrink inventory, get paid by customers faster, and pay suppliers later. This generates cash, reduces the need for raised capital, and creates a high return on capital.
*   **Focus on the product:** Follow "deep curiosity" to take an idea from your head and turn it into an actual product that makes "somebody else's life better."
*   **Energy management:** Focus on "energy management" rather than just time management to accomplish goals.

**What the goal should be:**
*   **Transformation:** To transform the business to be "faster, more efficient, [and] more capable."
*   **Service:** To "keep serving our customers" regardless of capital markets.
*   **Maximum

In [61]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [62]:

def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [63]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [66]:
parallel_chain.invoke('who is dell')

{'context': 'access to incredible tools, and so we\'re doing this\nall over our business. And of course, our customers are\ndoing it. And what is this doing? It\'s unlocking the power of data, using\ncompute and using all these models, which also happens to be, you know, we\'re in the business of compute,\nand storage, and networking. So, it\'s an enormous opportunity\nfor Dell itself to be able to help all of our customers unlock\nthe power of their data. - You couldn\'t predict, like, the actual\ntools that were going to be invented. You just knew that\nnow we have this new... There\'s this new technology. That it will transform every single\nprocess inside the business. We won\'t be able to predict all the tools,\nbut we just know that they\'re coming. - If we didn\'t do that,\nand our competitors did, we\'d be finished, all right? And so, we\'re not going to do that. And, you know, even if we did it\n\nor Are You Playing to Win?" It is a book about\nextreme winners and some of the 

In [67]:
parser = StrOutputParser()

In [69]:
main_chain = parallel_chain | prompt | model | parser

In [70]:
main_chain.invoke("how grow in bussiness?")

'Based on the transcript, you can grow a business by:\n\n*   **Implementing a negative cash conversion cycle:** This involves dramatically shrinking inventory, getting paid by customers faster, and paying suppliers later. This generates cash as you grow and reduces the need to raise outside capital.\n*   **Avoiding elongated supply chains:** Instead of using distributors and dealers like competitors, sell more directly to customers.\n*   **Scaling operations:** Organizing "uncontrolled chaos" and setting up the business for 10X or 100X growth.\n*   **Securing capital and credit:** Obtaining credit lines, private placements, or going public to provide the balance sheet needed to support fast growth.\n*   **Focusing on cost efficiency:** Taking out costs and finding inexpensive ways to win in your space, similar to retailers like Walmart, Costco, and Amazon.\n*   **Bringing in experienced help:** The speaker mentions how Lee Walker helped fill in "missing pieces," organize operations, an